# Cargar criterios de Excel a PostgreSQL

Carga los 397 registros de `criterios.xlsx`, hoja `criterios_centrales`, a
`public.criterios`: 331 perfiles y 66 principales para 66 carreras.

Las cinco columnas `carrera_id`, `type_id`, `riasec_id`, `desc_criterio` y
`resumen_criterio` conservan sus nombres. `origen` se guarda en `type_extraction`.
PostgreSQL genera `criterio_id` automáticamente.

Ejecuta las celdas en orden. La última celda **confirma la carga en la base de datos**.
La conexión utiliza `lib/set_envs.py` sin copiar credenciales al notebook.
Dependencias del kernel: `openpyxl` y `psycopg2-binary`.

In [ ]:
from collections import Counter
from pathlib import Path
import sys

import openpyxl
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'Scripts/python/jupyter/lib/set_envs.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Abre el notebook desde un directorio del repositorio Athena.')
sys.path.insert(0, str(ROOT / 'Scripts/python/jupyter'))
from lib.set_envs import DB_CONFIG

EXCEL_PATH = ROOT / 'Scripts/python/assets/layouts/excel/criterios.xlsx'
SHEET_NAME = 'criterios_centrales'
EXCEL_COLUMNS = ('carrera_id', 'type_id', 'riasec_id', 'desc_criterio',
                 'resumen_criterio', 'origen')
DB_COLUMNS = (*EXCEL_COLUMNS[:5], 'type_extraction')
EXPECTED_ROWS = 397

## Leer y validar el archivo

Se rechazan IDs inválidos, claves repetidas, fórmulas y textos vacíos.
No se recortan ni truncan las descripciones. Se comprueba el conjunto acordado:
66 carreras, 331 perfiles, 66 principales y origen `PERFIL` en todos los registros.

In [3]:
def leer_criterios(path=EXCEL_PATH):
    workbook = openpyxl.load_workbook(path, read_only=True, data_only=False)
    try:
        sheet = workbook[SHEET_NAME]
        rows = sheet.iter_rows(values_only=True)
        headers = tuple(next(rows))
        if headers != EXCEL_COLUMNS:
            raise ValueError(f'Columnas inesperadas: {headers}')
        records = []
        keys = set()
        for row_number, row in enumerate(rows, start=2):
            if all(value is None for value in row):
                continue
            row = tuple(row)
            for value in row[:3]:
                if type(value) is not int or value < 0:
                    raise ValueError(f'Fila {row_number}: ID inválido {value!r}')
            for value in row[3:]:
                if not isinstance(value, str) or not value.strip() or value.startswith('=') or '\x00' in value:
                    raise ValueError(f'Fila {row_number}: texto vacío o inválido')
            if len(row[4]) > 100:
                raise ValueError(f'Fila {row_number}: resumen excede VARCHAR(100)')
            if row[5] != 'PERFIL':
                raise ValueError(f'Fila {row_number}: origen distinto de PERFIL')
            key = row[:3]
            if key in keys:
                raise ValueError(f'Fila {row_number}: clave duplicada {key}')
            keys.add(key)
            records.append(row)
    finally:
        workbook.close()
    if len(records) != EXPECTED_ROWS:
        raise ValueError(f'Se esperaban {EXPECTED_ROWS} registros, se leyeron {len(records)}')
    if {r[0] for r in records} != set(range(1, 67)):
        raise ValueError('Se esperaban las carreras 1 a 66')
    if Counter(r[1] for r in records) != {1: 331, 2: 66}:
        raise ValueError('Distribución de tipos inesperada')
    for carrera_id in range(1, 67):
        career = [r for r in records if r[0] == carrera_id]
        if [r[2] for r in career if r[1] == 2] != [0]:
            raise ValueError(f'Carrera {carrera_id}: principal inválido')
        dimensions = {r[2] for r in career if r[1] == 1}
        if not {1, 2, 4, 5, 6} <= dimensions <= {1, 2, 3, 4, 5, 6}:
            raise ValueError(f'Carrera {carrera_id}: dimensiones inválidas')
    return records

registros = leer_criterios()
print(f'Excel validado: {len(registros)} criterios de 66 carreras.')
print('Por tipo:', dict(Counter(r[1] for r in registros)))

Excel validado: 397 criterios de 66 carreras.
Por tipo: {1: 331, 2: 66}


## Validar catálogos y cargar en una transacción

Antes de insertar se revisan las tres llaves foráneas y el límite de longitud
del esquema real. Una clave `(carrera_id, type_id, riasec_id)` ya existente se
omite únicamente si sus seis valores coinciden. Si contiene datos diferentes
o aparece duplicada, la carga se detiene sin modificarla.

La carga bloquea brevemente escrituras concurrentes en `criterios`, inserta
con parámetros SQL y verifica cada registro antes del commit. Ante cualquier
error se revierte la transacción. `confirmar=False` solo consulta y valida.

In [4]:
def cargar_criterios(records, confirmar=False):
    if not records:
        raise ValueError('No hay criterios para cargar')
    expected = {r[:3]: tuple(r) for r in records}
    if len(expected) != len(records):
        raise ValueError('Hay claves duplicadas en la carga')
    conn = psycopg2.connect(**DB_CONFIG, connect_timeout=10)
    conn.set_session(readonly=not confirmar)
    try:
        with conn.cursor() as cur:
            cur.execute("SET LOCAL lock_timeout = '10s'")
            cur.execute("SET LOCAL statement_timeout = '60s'")
            if confirmar:
                cur.execute('LOCK TABLE public.criterios IN SHARE ROW EXCLUSIVE MODE')
            for index, table, column in (
                (0, 'carreras', 'carrera_id'),
                (1, 'tipos_criterio', 'type_id'),
                (2, 'tipos_riasec', 'riasec_id'),
            ):
                ids = {r[index] for r in records}
                query = sql.SQL('SELECT {} FROM public.{} WHERE {} = ANY(%s)').format(
                    sql.Identifier(column), sql.Identifier(table), sql.Identifier(column))
                cur.execute(query, (sorted(ids),))
                missing = ids - {r[0] for r in cur.fetchall()}
                if missing:
                    raise ValueError(f'IDs inexistentes en {table}: {sorted(missing)}')
            cur.execute("""SELECT column_name, character_maximum_length
                           FROM information_schema.columns
                           WHERE table_schema = 'public' AND table_name = 'criterios'""")
            limits = dict(cur.fetchall())
            if not set(DB_COLUMNS).issubset(limits):
                raise ValueError('El esquema de criterios no coincide con las columnas requeridas')
            for row in records:
                for column, value in zip(DB_COLUMNS, row):
                    if limits[column] is not None and isinstance(value, str) and len(value) > limits[column]:
                        raise ValueError(f'{row[:3]}: {column} excede {limits[column]} caracteres')
            select = sql.SQL('SELECT {} FROM public.criterios WHERE carrera_id = ANY(%s)').format(
                sql.SQL(', ').join(map(sql.Identifier, DB_COLUMNS)))
            career_ids = sorted({r[0] for r in records})
            cur.execute(select, (career_ids,))
            existing = {}
            for row in cur.fetchall():
                key = row[:3]
                if key not in expected:
                    continue
                if key in existing or row != expected[key]:
                    raise ValueError(f'Clave {key} duplicada o con contenido diferente en la base')
                existing[key] = row
            pending = [row for key, row in expected.items() if key not in existing]
            result = {'excel': len(records), 'existentes_identicos': len(existing),
                      'pendientes': len(pending), 'insertados': 0, 'confirmado': False}
            if not confirmar:
                conn.rollback()
                return result
            if pending:
                query = sql.SQL('INSERT INTO public.criterios ({}) VALUES %s').format(
                    sql.SQL(', ').join(map(sql.Identifier, DB_COLUMNS)))
                execute_values(cur, query.as_string(cur), pending, page_size=500)
            cur.execute(select, (career_ids,))
            actual = Counter(row for row in cur.fetchall() if row[:3] in expected)
            if actual != Counter(expected.values()):
                raise RuntimeError('La verificación no coincide con todos los registros del Excel')
            conn.commit()
            result.update(insertados=len(pending), confirmado=True)
            return result
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

In [5]:
validacion = cargar_criterios(registros, confirmar=False)
print(validacion)

{'excel': 397, 'existentes_identicos': 397, 'pendientes': 0, 'insertados': 0, 'confirmado': False}


## Ejecutar la carga

Esta celda vuelve a leer el Excel y repite las validaciones antes de insertar.
Una segunda ejecución conserva los registros idénticos y reporta cero inserciones.

In [6]:
registros = leer_criterios()
resultado = cargar_criterios(registros, confirmar=True)
print(resultado)
verificacion = cargar_criterios(registros, confirmar=False)
if verificacion['pendientes'] != 0 or verificacion['existentes_identicos'] != len(registros):
    raise RuntimeError(f'Verificación posterior al commit fallida: {verificacion}')
print(f"Carga verificada: {verificacion['existentes_identicos']} criterios coinciden con el Excel.")

{'excel': 397, 'existentes_identicos': 397, 'pendientes': 0, 'insertados': 0, 'confirmado': True}
Carga verificada: 397 criterios coinciden con el Excel.
